In [47]:
# JASNY GWINT!

In [48]:
import json

records = []
# Load from a file
with open("../data/ait_ads/fox_wazuh.json", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:  # skip blank lines
            record = json.loads(line)
            if not "Suricata" in record["rule"]["description"] and not "suricata" in record["rule"]["groups"]:
                records.append(record)

print(records[0])

{'predecoder': {'hostname': 'mail', 'program_name': 'freshclam', 'timestamp': 'Jan 15 02:32:32'}, 'agent': {'ip': '172.17.131.81', 'name': 'wazuh-client', 'id': '18'}, 'manager': {'name': 'wazuh.manager'}, 'rule': {'firedtimes': 1, 'mail': False, 'level': 3, 'pci_dss': ['5.2'], 'tsc': ['A1.2'], 'description': 'ClamAV database update', 'groups': ['clamd', 'freshclam', 'virus'], 'id': '52507', 'nist_800_53': ['SI.3'], 'gpg13': ['4.4'], 'gdpr': ['IV_35.7.d']}, 'decoder': {'name': 'freshclam'}, 'full_log': 'Jan 15 02:32:32 mail freshclam[29266]: Sat Jan 15 02:32:32 2022 -> ClamAV update process started at Sat Jan 15 02:32:32 2022', 'input': {'type': 'log'}, '@timestamp': '2022-01-15T02:32:32.000000Z', 'location': '/var/log/syslog', 'id': '1686147193.86593'}


In [49]:
import pandas as pd
df = pd.json_normalize(records)
print(df.shape)
df.head()

(451439, 60)


,full_log,@timestamp,location,id,predecoder.hostname,predecoder.program_name,predecoder.timestamp,agent.ip,agent.name,agent.id,...,data.uid,data.srcuser,data.tty,data.pwd,data.command,decoder.ftscomment,data.protocol,data.url,data.euid,previous_output
0,Jan 15 02:32:32 mail freshclam[29266]: Sat Jan...,2022-01-15T02:32:32.000000Z,/var/log/syslog,1686147193.86593,mail,freshclam,Jan 15 02:32:32,172.17.131.81,wazuh-client,18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Jan 15 02:32:32 taylorcruz-mail freshclam[2851...,2022-01-15T02:32:32.000000Z,/var/log/syslog,1686147193.86958,taylorcruz-mail,freshclam,Jan 15 02:32:32,192.168.128.170,wazuh-client,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Jan 15 02:32:37 mail freshclam[29266]: Sat Jan...,2022-01-15T02:32:37.000000Z,/var/log/syslog,1686147197.87335,mail,freshclam,Jan 15 02:32:37,172.17.131.81,wazuh-client,18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Jan 15 02:32:42 mail freshclam[29266]: Sat Jan...,2022-01-15T02:32:42.000000Z,/var/log/syslog,1686147203.87700,mail,freshclam,Jan 15 02:32:42,172.17.131.81,wazuh-client,18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Jan 15 02:32:47 mail freshclam[29266]: Sat Jan...,2022-01-15T02:32:47.000000Z,/var/log/syslog,1686147207.88065,mail,freshclam,Jan 15 02:32:47,172.17.131.81,wazuh-client,18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:
def print_unique_values(df):
    for col in df.columns:
        try:
            counts = df[col].value_counts(dropna=False)
        except TypeError:  # lists or dicts aren't hashable
            counts = df[col].astype(str).value_counts(dropna=False)

        print(f"\n=== {col} ({counts.shape[0]} unique) ===")
        print(counts)

print_unique_values(df)


=== full_log (432528 unique) ===
full_log
01/19/2022-01:36:58.942284  [**] [1:2013504:6] ET POLICY GNU/Linux APT User-Agent Outbound likely related to package management [**] [Classification: Not Suspicious Traffic] [Priority: 3] {TCP} 192.168.128.4:49172 -> 91.189.88.142:80    5
01/19/2022-02:34:27.644771  [**] [1:2013504:6] ET POLICY GNU/Linux APT User-Agent Outbound likely related to package management [**] [Classification: Not Suspicious Traffic] [Priority: 3] {TCP} 192.168.128.4:42720 -> 91.189.88.142:80    5
Jan 15 06:26:59 mail dovecot: imap-login: Login: user=<daniel.morgan>, method=PLAIN, rip=172.17.131.81, lip=172.17.131.81, mpid=9147, TLS, session=<AaW8AZnV5LmsEYNR>                                                                        3
Jan 15 06:26:59 mail dovecot: imap-login: Login: user=<daniel.morgan>, method=PLAIN, rip=172.17.131.81, lip=172.17.131.81, mpid=9150, TLS, session=<blK9AZnV5rmsEYNR>                                                                        3
J

**Your feature set has a likely leakage problem: several of the `rule.*` fields are derived from the same rules that produced the label, so the model may learn the rules rather than the attacks.** Before training, check how `attack` was labeled. If it came from Wazuh rule levels, groups, or MITRE mappings, those features will make the task trivially easy and the model won't generalize to new traffic.

Here's how I'd group your features:

| Group | Features | Recommendation |
|---|---|---|
| **Likely leakage** | `rule.level`, `rule.groups`, `rule.mitre.*`, `rule.pci_dss`, `rule.tsc`, `rule.nist_800_53`, `rule.gpg13`, `rule.gdpr`, `rule.hipaa` | Drop these if the label depends on Wazuh's rules. The compliance fields are just rule metadata, so they mostly identify the rule that fired. |
| **Identifiers** | `id`, `data.id`, `full_log` (as raw text) | Drop `id` since it's unique per alert and carries no signal. `data.id` is probably a user or session identifier and risks memorization. `full_log` needs parsing or vectorization, not direct use. |
| **Time** | `@timestamp` | Don't feed raw timestamps. Extract hour of day, day of week, and possibly inter-event gaps per source IP. |
| **Network** | `data.srcip`, `data.dstip`, `data.srcport`, `data.protocol`, `data.url` | Keep these but encode carefully. IPs are high cardinality, so use derived features like counts per IP over a time window, whether the IP is internal or external, or port categories. |
| **Identity and session** | `data.srcuser`, `data.dstuser`, `data.tty` | Usually sparse. Consider flags such as "login as root" or "failed attempts per user" instead of raw values. |
| **Geo** | `GeoLocation.*` | Mostly populated only for external IPs. Consider a missing-value indicator, and beware that country-level features can encode the data source more than attack behavior. |
| **Source metadata** | `location`, `predecoder.*`, `decoder.name`, `decoder.parent` | These describe which log produced the event. They're useful for context but can cause the model to learn log-source imbalances. |
| **Unclear** | `previous_output` | I can't tell what this is from the name. If it's the output of an earlier pipeline stage or a prior model, check that it isn't built from the label. |

Two other things to check. First, time-based splits matter more than random splits here, since attack patterns drift and random splits can inflate your scores. Second, sparse fields like `data.url` and the `GeoLocation` fields may be mostly null, so confirm their null rates before relying on them.

What does the `attack` label come from, and what's `previous_output`? With those answers I can say more precisely which features to keep.

In [59]:
df = df.drop(columns=[
    "agent.name", "agent.id", "agent.ip", "id",
    "manager.name",
    "rule.firedtimes", "rule.mail", "rule.description", "rule.id", "rule.frequency", "rule.level", "rule.groups", "rule.mitre.technique", "rule.mitre.id", "rule.mitre.tactic", "rule.pci_dss", "rule.tsc", "rule.nist_800_53", "rule.gpg13", "rule.gdpr", "rule.hipaa", # zastąpić przez fired times ale w oknie
    "predecoder.timestamp",
    "input.type",
    "data.audit.uid", "data.audit.auid", "data.audit.gid", "data.audit.exe", "data.audit.euid", "data.audit.session", "data.audit.pid", "data.audit.id", "data.audit.type", "data.audit.command", "data.uid", "data.pwd", "data.command", "data.euid", "data.id", # wyglądają bezurzytecznie
    "decoder.ftscomment", "decoder.name", "decoder.parent",
    "GeoLocation.city_name", "GeoLocation.country_name", "GeoLocation.location.lat", "GeoLocation.location.lon", "GeoLocation.region_name", "location",
    "predecoder.hostname", 'predecoder.program_name',
    "previous_output"
], errors="ignore")

# jeżeli location = "/var/log/apache2/intranet-access.log" to host_role = WWW
# dstip oraz srcip zamienić na wew albo zew
# data.srcuser zamienić na is_known_user


In [61]:
print_unique_values(df)
print(df.shape)


=== full_log (432528 unique) ===
full_log
01/19/2022-01:36:58.942284  [**] [1:2013504:6] ET POLICY GNU/Linux APT User-Agent Outbound likely related to package management [**] [Classification: Not Suspicious Traffic] [Priority: 3] {TCP} 192.168.128.4:49172 -> 91.189.88.142:80    5
01/19/2022-02:34:27.644771  [**] [1:2013504:6] ET POLICY GNU/Linux APT User-Agent Outbound likely related to package management [**] [Classification: Not Suspicious Traffic] [Priority: 3] {TCP} 192.168.128.4:42720 -> 91.189.88.142:80    5
Jan 15 06:26:59 mail dovecot: imap-login: Login: user=<daniel.morgan>, method=PLAIN, rip=172.17.131.81, lip=172.17.131.81, mpid=9147, TLS, session=<AaW8AZnV5LmsEYNR>                                                                        3
Jan 15 06:26:59 mail dovecot: imap-login: Login: user=<daniel.morgan>, method=PLAIN, rip=172.17.131.81, lip=172.17.131.81, mpid=9150, TLS, session=<blK9AZnV5rmsEYNR>                                                                        3
J

In [60]:
df.columns

Index(['full_log', '@timestamp', 'data.srcip', 'data.dstip', 'data.dstuser',
       'data.srcport', 'data.srcuser', 'data.tty', 'data.protocol',
       'data.url'],
      dtype='str')

# Generowanie dodatkowych danych

In [64]:
import numpy as np
df["is_dst_user_root"] = np.where(df["data.dstuser"] == "root", 1, 0)
# data.srcip -> ip wew lub zew
# data.dstip -> ip wew lub zew
# full_log -> wektoryzacja
# @timestamp -> godzina_dnia, dzień_tygodnia, czas_między_ostatnim_tym_samym_srcip
# data.srcport -> kategoria portu (SSH - 22, etc)


In [ ]:
# planowane

def entropy_of(text: str) -> float:
    if not text:
        return 0.0
    counts = Counter(text)
    total = len(text)
    value = 0.0
    for count in counts.values():
        probability = count / total
        value -= probability * math.log2(probability)
    return round(value, 4)

def url_features(url: object, scan_text: str) -> dict[str, float]:
    text = as_str(url)
    lowered = text.lower()
    out: dict[str, float] = {
        "url_len": float(len(text)),
        "url_parts": float(len([part for part in lowered.split("/") if part])),
        "url_query": float("?" in text),
        "url_tokens": float(len(re.findall(r"[A-Za-z0-9]+", lowered))),
        "url_encoded_chars": float(lowered.count("%")),
        "url_entropy": entropy_of(text),
        "url_max_token_len": float(max((len(t) for t in TOKEN_RX.findall(text)), default=0)),
    }
    for flag, needle in URL_FLAGS.items():
        out[flag] = float(needle in lowered)
    low_scan = (scan_text or "").lower()
    for flag, needle in UA_PATTERNS.items():
        # needles are case-insensitive here; low_scan is already lowercased
        out[flag] = float(needle.lower() in low_scan)
    return out

def scan_text_for_url(raw: str) -> tuple[str, str, str]:
    match = REQ_RX.search(raw)
    if match:
        return match.group(2), match.group(1), match.group(3)
    return "", "", ""


def log_resource_class(resources: list) -> str:
    text = " ".join(as_str(r) for r in resources).lower()
    if "access" in text or "apache" in text:
        return "apache"
    if "metricbeat" in text or "filebeat" in text:
        return "metrics"
    if "syslog" in text or "auth" in text:
        return "syslog"
    return "other"